# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata info via object attributes
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's enumerate the available record sets, the fields in each, and their corresponding `@id`s.

In [ ]:
from pprint import pprint

# List RecordSets
record_sets = dataset.metadata.record_sets
print(f"Record Sets (by @id):\n----------------------")
for rs in record_sets:
    print(f"@id: {rs.id}, name: {rs.name if hasattr(rs, 'name') else ''}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for fld in rs.fields:
            print(f"    @id: {fld.id}, name: {fld.name if hasattr(fld, 'name') else ''}, dataType: {fld.data_type if hasattr(fld, 'data_type') else ''}")
    print()

# For demonstration, select the first record set
if record_sets:
    selected_record_set = record_sets[0].id
    print(f"Selected Record Set @id for examples: {selected_record_set}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set
dataframes = {}

# Use all record sets
all_record_set_ids = [rs.id for rs in record_sets]
for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show the columns of the first (selected) record set
sel_id = selected_record_set
if sel_id in dataframes:
    print(f"Columns in record set {sel_id}:")
    print(dataframes[sel_id].columns.tolist())
    display(dataframes[sel_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes.

We'll automatically detect a numeric field from the record set's fields. _You can substitute your target field/grouping as you explore the metadata above._

In [ ]:
# Select a numeric field by searching field data types in the selected record set
import numpy as np

sel_rs = None
for rs in record_sets:
    if rs.id == sel_id:
        sel_rs = rs
        break

numeric_field_id = None
if sel_rs:
    for fld in sel_rs.fields:
        if hasattr(fld, 'data_type') and fld.data_type in ['Float', 'Integer', 'Number']:
            numeric_field_id = fld.id
            break

if numeric_field_id is None:
    # Fallback to a column with numeric content
    for col in dataframes[sel_id].columns:
        if pd.api.types.is_numeric_dtype(dataframes[sel_id][col]):
            numeric_field_id = col
            break

print(f"Using numeric field: {numeric_field_id}")
# Remove records with missing/nonnumeric
df = dataframes[sel_id]
if numeric_field_id not in df.columns:
    print(f'Numeric field {numeric_field_id} not found in DataFrame columns!')
else:
    threshold = df[numeric_field_id].quantile(0.75) if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
    ) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()

    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Attempt to group by a categorical field if present
    group_field = None
    if sel_rs:
        for fld in sel_rs.fields:
            if hasattr(fld, 'data_type') and fld.data_type in ['Text', 'String'] and fld.id in df.columns and df[fld.id].nunique() > 1:
                group_field = fld.id
                break
    if group_field is None:
        for col in df.columns:
            if df[col].dtype == "object" and df[col].nunique() > 1:
                group_field = col
                break
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Let's plot the distribution of the selected numeric field and show the mean values by group if grouping was performed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

# Plot group means if available
if 'grouped_df' in locals() and group_field:
    plt.figure(figsize=(10,5))
    grouped_vals = grouped_df[numeric_field_id] if numeric_field_id in grouped_df.columns else grouped_df.iloc[:,0]
    grouped_vals.plot(kind='bar')
    plt.title(f'Mean {numeric_field_id} by {group_field}')
    plt.xlabel(group_field)
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded and explored the FAIR^2 mass spectrometry dataset via its Croissant schema.
- Record sets and fields can be referenced by their Croissant `@id`.
- Extracted tabular data and performed simple exploratory filtering, normalization, and aggregation.
- Visualized the distribution of a numeric field and (if available) group-wise means.

This provides a starting point for deeper domain-specific analysis!